In [1]:
import tensorflow as tf
from tensorflow import keras
from keras import layers, models
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import time
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from keras.callbacks import EarlyStopping,ReduceLROnPlateau
from matplotlib import pyplot as plt

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"keshrivishal21","key":"4c8bb86460f4dd386e694fd83bc26447"}'}

In [3]:
!mkdir -p ~/.config/kaggle
!mv kaggle.json ~/.config/kaggle/
!chmod 600 ~/.config/kaggle/kaggle.json

In [4]:
!kaggle competitions download -c state-farm-distracted-driver-detection

100% 4.00G/4.00G [00:49<00:00, 87.5MB/s]



In [5]:
!unzip -q state-farm-distracted-driver-detection.zip -d data

In [6]:
df = pd.read_csv("/content/data/driver_imgs_list.csv")
df.head()

,subject,classname,img
0,p002,c0,img_44733.jpg
1,p002,c0,img_72999.jpg
2,p002,c0,img_25094.jpg
3,p002,c0,img_69092.jpg
4,p002,c0,img_92629.jpg


In [7]:
drivers = df['subject'].unique()
print("Total drivers:", len(drivers))

Total drivers: 26


In [8]:
train_drivers, temp_drivers = train_test_split(
    drivers,
    test_size=0.4,
    random_state=42
)

val_drivers, test_drivers = train_test_split(
    temp_drivers,
    test_size=0.5,
    random_state=42
)

print("Train drivers:", len(train_drivers))
print("Val drivers:", len(val_drivers))
print("Test drivers:", len(test_drivers))

Train drivers: 15
Val drivers: 5
Test drivers: 6


In [9]:
train_df = df[df['subject'].isin(train_drivers)]
val_df = df[df['subject'].isin(val_drivers)]
test_df = df[df['subject'].isin(test_drivers)]

print(train_df['subject'].nunique())
print(val_df['subject'].nunique())
print(test_df['subject'].nunique())

15
5
6


In [10]:
base_path = "/content/data/imgs/train"

def add_image_path(df):
    return df.apply(
        lambda row: os.path.join(base_path, row['classname'], row['img']),
        axis=1
    )

train_paths = add_image_path(train_df)
val_paths = add_image_path(val_df)
test_paths = add_image_path(test_df)

train_labels = train_df['classname']
val_labels = val_df['classname']
test_labels = test_df['classname']

In [11]:
le = LabelEncoder()
train_labels = le.fit_transform(train_labels)
val_labels = le.transform(val_labels)
test_labels = le.transform(test_labels)

In [12]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
tf.random.set_seed(42)

def process_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)

    h = tf.shape(image)[0]
    w = tf.shape(image)[1]

    # 🔥 random jitter
    shift_h = tf.random.uniform([], -0.05, 0.05)
    shift_w = tf.random.uniform([], -0.05, 0.05)

    top = tf.cast((0.40 + shift_h) * tf.cast(h, tf.float32), tf.int32)
    bottom = tf.cast(0.95 * tf.cast(h, tf.float32), tf.int32)
    left = tf.cast((0.25 + shift_w) * tf.cast(w, tf.float32), tf.int32)
    right = tf.cast(0.80 * tf.cast(w, tf.float32), tf.int32)

    roi = image[top:bottom, left:right]

    image = tf.image.resize(image, IMG_SIZE)
    roi = tf.image.resize(roi, IMG_SIZE)

    image = tf.cast(image, tf.float32)
    roi = tf.cast(roi, tf.float32)

    image = tf.clip_by_value(image, 0.0, 255.0)
    roi = tf.clip_by_value(roi, 0.0, 255.0)

    return (image, roi), label

def add_noise(image, roi, label):
    image = image + tf.random.normal(tf.shape(image), 0.0, 0.8)
    roi = roi + tf.random.normal(tf.shape(roi), 0.0, 0.8)
    return (image, roi), label

def create_dataset(paths, labels, shuffle=True, training=False):
    labels = tf.convert_to_tensor(labels, dtype=tf.int32)
    labels = tf.one_hot(labels, depth=10)

    paths = paths.values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    # ✅ Shuffle BEFORE mapping
    if shuffle:
        ds = ds.shuffle(buffer_size=1500, reshuffle_each_iteration=True)

    # Load & preprocess
    ds = ds.map(process_image, num_parallel_calls=tf.data.AUTOTUNE)

    # ✅ Add noise ONLY for training
    if training:
        ds = ds.map(
            lambda x, y: add_noise(x[0], x[1], y),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    # Batch
    ds = ds.batch(BATCH_SIZE, drop_remainder=True)

    # Prefetch
    ds = ds.prefetch(tf.data.AUTOTUNE)

    # Performance tweak
    options = tf.data.Options()
    options.experimental_deterministic = False
    ds = ds.with_options(options)

    return ds

train_ds = create_dataset(train_paths, train_labels, shuffle=True)
val_ds = create_dataset(val_paths, val_labels, shuffle=False)
test_ds = create_dataset(test_paths, test_labels, shuffle=False)

In [13]:
for path in train_paths[:5]:
    print(path, os.path.exists(path))

/content/data/imgs/train/c0/img_48693.jpg True
/content/data/imgs/train/c0/img_44903.jpg True
/content/data/imgs/train/c0/img_58514.jpg True
/content/data/imgs/train/c0/img_62307.jpg True
/content/data/imgs/train/c0/img_83984.jpg True


In [14]:
set(train_drivers) & set(val_drivers)
set(train_drivers) & set(test_drivers)
set(val_drivers) & set(test_drivers)

set()

In [15]:
for images, labels in train_ds.take(1):
    print("Full Image batch shape:", images[0].shape)
    print("ROI Image batch shape:", images[1].shape)
    print("Label batch shape:", labels.shape)

Full Image batch shape: (16, 224, 224, 3)
ROI Image batch shape: (16, 224, 224, 3)
Label batch shape: (16, 10)


In [16]:
print("Train images:", len(train_paths))
print("Val images:", len(val_paths))
print("Test images:", len(test_paths))

Train images: 13440
Val images: 3312
Test images: 5672


In [17]:
def dual_input_model(input_shape=(224,224,3), num_classes=10):

    # Inputs
    input_full = layers.Input(shape=input_shape)
    input_roi = layers.Input(shape=input_shape)

    # Augmentation
    aug = data_augmentation

    x1 = aug(input_full)
    x2 = aug(input_roi)

    # Preprocessing
    x1 = tf.keras.applications.efficientnet.preprocess_input(x1)
    x2 = tf.keras.applications.efficientnet.preprocess_input(x2)

    # Shared backbone (IMPORTANT)
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape
    )

    base_model.trainable = False

    f1 = base_model(x1, training=False)
    f2 = base_model(x2, training=False)

    # Pooling
    f1 = layers.GlobalAveragePooling2D()(f1)
    f2 = layers.GlobalAveragePooling2D()(f2)

    x = layers.Concatenate()([f1, f2])
    x = layers.Dense(256, activation='relu',kernel_regularizer=tf.keras.regularizers.l2(1e-3))(x)
    x = layers.Dense(128, activation='relu',kernel_regularizer=tf.keras.regularizers.l2(1e-3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.6)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model([input_full, input_roi], outputs)

    return model, base_model

In [18]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

In [19]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.08),
    layers.RandomContrast(0.1),
])

In [20]:
model, base_model = dual_input_model()
model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 224, 224,  │          0 │ input_layer[0][0… │
│ (Sequential)        │ 3)                │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetb0      │ (None, 7, 7,      │  4,049,571 │ sequential[0][0], │
│ (Functional)        │ 1280)             │            │ sequential[1][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ efficientnetb0[0… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ efficientnetb0[1… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 2560)      │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    655,616 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128)       │        512 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 10)        │      1,290 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,739,885 (18.08 MB)

 Trainable params: 690,058 (2.63 MB)

 Non-trainable params: 4,049,827 (15.45 MB)

In [21]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.15),
    metrics=['accuracy']
)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 124s 118ms/step - accuracy: 0.4940 - loss: 2.4481 - val_accuracy: 0.4912 - val_loss: 2.3224 - learning_rate: 1.0000e-04
Epoch 2/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 93s 110ms/step - accuracy: 0.7500 - loss: 1.8483 - val_accuracy: 0.5121 - val_loss: 2.3454 - learning_rate: 1.0000e-04
Epoch 3/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 93s 111ms/step - accuracy: 0.8461 - loss: 1.6549 - val_accuracy: 0.6244 - val_loss: 2.0721 - learning_rate: 1.0000e-04
Epoch 4/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 93s 110ms/step - accuracy: 0.8965 - loss: 1.5400 - val_accuracy: 0.5332 - val_loss: 2.1539 - learning_rate: 1.0000e-04
Epoch 5/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 92s 110ms/step - accuracy: 0.9205 - loss: 1.4579 - val_accuracy: 0.6045 - val_loss: 1.9906 - learning_rate: 1.0000e-04
Epoch 6/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 92s 110ms/step - accuracy: 0.9388 - loss: 1.3842 - val_accuracy: 0.5936 - val_loss: 1.9867 - learning_rate: 1.0000e-04
Epoch 7/30
840/840 ━━━━━━━━━━━━━━━━━━━━ 92s 1

In [23]:
test_loss, test_acc = model.evaluate(test_ds)
print("Driver-wise val Accuracy:", test_acc)

354/354 ━━━━━━━━━━━━━━━━━━━━ 33s 92ms/step - accuracy: 0.6118 - loss: 1.6516
Driver-wise val Accuracy: 0.6117584705352783
